# Atelier 1.A — Prise en main

**Durée : 20 min**

Objectifs :
- vérifier que l'environnement et les accès fonctionnent ;
- faire un premier appel avec le SDK `openai`, utilisé ici pour **tous** les fournisseurs : Groq, NVIDIA, OpenRouter et Hugging Face ;
- lire la réponse : texte, raison d'arrêt, consommation de tokens ;
- passer des tokens au coût, et comprendre les limites des paliers gratuits ;
- afficher une réponse en *streaming* ;
- découvrir la couche commune `outils_llm`, utilisée dans les ateliers suivants.

> Avant de commencer : copiez `.env.example` en `.env` à la racine du dépôt et renseignez au moins une clé (Groq ou NVIDIA conseillés).

## 1. Vérifier l'environnement

In [ ]:
import sys
import outils_llm as llm

print("Python", sys.version.split()[0])
disponibles = llm.fournisseurs_disponibles()
for fournisseur, ok in disponibles.items():
    print("OK " if ok else "-- ", fournisseur)

MODELE = llm.choisir_modele()
print("\nModèle utilisé dans cet atelier :", MODELE)

## 2. Premier appel

Groq, NVIDIA, OpenRouter et Hugging Face exposent le même format d'API, celui d'OpenAI (*Chat Completions*). Un seul SDK suffit : on change l'adresse du service (`base_url`), la clé et le nom du modèle.

In [ ]:
import os
from openai import OpenAI

fournisseur, nom_modele = llm.decouper(MODELE)
url, variable_cle = llm.FOURNISSEURS[fournisseur]
client = OpenAI(base_url=url, api_key=os.getenv(variable_cle))

reponse = client.chat.completions.create(
    model=nom_modele,
    max_tokens=2000,
    messages=[
        {"role": "system", "content": "Tu es l'assistant du service client de Fictiva SA. Réponds en français, en deux phrases au plus."},
        {"role": "user", "content": "Quels sont vos délais de livraison habituels ?"},
    ],
)
print(reponse.choices[0].message.content)
print("\nRaison d'arrêt :", reponse.choices[0].finish_reason)
print("Tokens :", reponse.usage.prompt_tokens, "en entrée,", reponse.usage.completion_tokens, "en sortie")

**À observer**

- `choices[0].message.content` contient le texte ; `finish_reason` vaut `stop` si la réponse est complète, `length` si elle a été coupée par `max_tokens`.
- gpt-oss est un **modèle de raisonnement** : il réfléchit avant de répondre. Ces tokens de raisonnement sont comptés dans `completion_tokens`, d'où un nombre de tokens de sortie supérieur à la longueur visible de la réponse.
- Le modèle ne connaît pas Fictiva : les délais qu'il annonce sont **inventés**. C'est une hallucination. Le module 1.3 et le Jour 2 (RAG) traitent ce problème.

## 3. Le même modèle chez plusieurs fournisseurs

gpt-oss-20b est un modèle open-weight : plusieurs fournisseurs le servent, avec des latences, des tarifs et des quotas différents. La boucle ci-dessous interroge tous ceux que vous avez configurés.

In [ ]:
import time

MEME_MODELE = {
    "groq": "openai/gpt-oss-20b",
    "nvidia": "openai/gpt-oss-20b",
    "huggingface": "openai/gpt-oss-20b",
    "openrouter": "openai/gpt-oss-20b",   # version payante : nécessite des crédits OpenRouter
}
question = [{"role": "user", "content": "En une phrase : qu'est-ce qu'un token ?"}]

for fournisseur, nom in MEME_MODELE.items():
    if not disponibles[fournisseur]:
        continue
    debut = time.perf_counter()
    try:
        # délai court, sans nouvelle tentative : un palier gratuit saturé ne bloque pas la cellule
        r = llm.client(fournisseur).with_options(timeout=45, max_retries=0).chat.completions.create(
            model=nom, messages=question, max_tokens=1000)
        print(f"{fournisseur:12} {time.perf_counter() - debut:5.1f} s  {r.choices[0].message.content[:120]}")
    except Exception as erreur:
        print(f"{fournisseur:12} erreur : {type(erreur).__name__} {str(erreur)[:120]}")

## 4. Compter les tokens

Le coût, la place occupée dans la fenêtre de contexte et les quotas se mesurent en **tokens**, pas en mots.
Comparez le même message en français, en anglais et en JSON. Le compte inclut l'enveloppe du message (quelques tokens) : comparez les écarts plutôt que les valeurs absolues.

In [ ]:
TEXTES = {
    "français": "Bonjour, nous souhaitons retourner trois claviers commandés par erreur et être remboursés.",
    "anglais": "Hello, we would like to return three keyboards ordered by mistake and get a refund.",
    "json": '{"categorie": "remboursement", "urgence": "normale", "produits": [{"reference": "KB-80", "quantite": 3}]}',
}

for nom, texte in TEXTES.items():
    r = client.chat.completions.create(model=nom_modele, messages=[{"role": "user", "content": texte}], max_tokens=1)
    print(f"{nom:9} {len(texte):4} caractères -> {r.usage.prompt_tokens} tokens")

## 5. Des tokens au coût

Les paliers gratuits ne sont pas facturés, mais ils sont **limités** (requêtes et tokens par minute et par jour) et réservés au prototypage. Pour un passage en production, on estime le coût au tarif payant :

coût d'un appel = tokens en entrée × prix d'entrée + tokens en sortie × prix de sortie.

Les tarifs publics indicatifs (USD par million de tokens) sont dans `llm.TARIFS`. Fictiva reçoit 300 demandes par jour ouvré (22 jours par mois) ; le tri d'une demande consomme environ 900 tokens en entrée et 250 en sortie.

In [ ]:
import pandas as pd

DEMANDES_PAR_JOUR = 300
JOURS_OUVRES = 22
TOKENS_ENTREE = 900   # prompt système + email
TOKENS_SORTIE = 250   # réponse structurée, raisonnement compris

def cout_mensuel(nom_modele):
    prix_entree, prix_sortie = llm.TARIFS[nom_modele]
    par_demande = (TOKENS_ENTREE * prix_entree + TOKENS_SORTIE * prix_sortie) / 1_000_000
    return round(par_demande * DEMANDES_PAR_JOUR * JOURS_OUVRES, 2)

pd.DataFrame({"coût mensuel (USD)": {nom: cout_mensuel(nom) for nom in llm.TARIFS}}).sort_values("coût mensuel (USD)")

**Questions**
1. Refaites le calcul avec 1 500 tokens de sortie (raisonnement plus long). Quel paramètre pèse le plus ?
2. À titre de comparaison, un grand modèle propriétaire facturé 4 $ / 20 $ par million de tokens : combien par mois ?
3. Le palier gratuit de votre fournisseur suffirait-il pour 300 demandes par jour ? Consultez ses quotas (console du fournisseur).

## 6. Streaming

En *streaming*, le texte s'affiche au fil de la génération : meilleure expérience utilisateur et pas de délai d'expiration sur les longues réponses.

In [ ]:
flux = client.chat.completions.create(
    model=nom_modele,
    messages=[{"role": "user", "content": "Rédige un court email (5 lignes) informant un client que sa commande CMD-2026-0412 sera livrée demain."}],
    stream=True,
)
for morceau in flux:
    if morceau.choices and morceau.choices[0].delta.content:
        print(morceau.choices[0].delta.content, end="", flush=True)

## 7. La couche commune `outils_llm`

Les ateliers suivants passent par `llm.demander(...)`, qui choisit le bon client, mesure chaque appel et gère les sorties structurées.
Un modèle se désigne par `fournisseur:modèle`, par exemple `groq:openai/gpt-oss-20b` ou `nvidia:nvidia/nemotron-3.5-lightning-30b-a3b`.

`demander` renvoie un objet `Reponse` : `texte`, `tokens_entree`, `tokens_sortie`, `latence_s`, `cout_usd` (coût estimé au tarif payant), `fin` (raison d'arrêt), `objet` et `structure` (sorties structurées, atelier 1.D).
En cas de dépassement de quota (erreur 429), le client attend le délai indiqué par le serveur puis réessaie.

In [ ]:
r = llm.demander("En une phrase : qu'est-ce qu'une fenêtre de contexte ?", modele=MODELE)
r

## 8. Une erreur fréquente

Un nom de modèle mal formé : il manque le fournisseur.

In [ ]:
try:
    llm.demander("Bonjour", modele="openai/gpt-oss-20b")
except ValueError as erreur:
    print("ValueError :", erreur)

## À retenir

- Un appel = un modèle, des messages, une limite de tokens en sortie. La réponse donne le texte, la raison d'arrêt et la consommation.
- Un même modèle open-weight est servi par plusieurs fournisseurs : on compare leurs latences, leurs prix et leurs quotas.
- Gratuit ne veut pas dire illimité : les paliers gratuits sont plafonnés et réservés au prototypage.
- Les clés restent dans `.env`, jamais dans un notebook ni dans Git.
- Sans contexte, le modèle invente : il faudra lui fournir les bonnes informations (module 1.3 et Jour 2).